In [1]:
import scanpy as sc
import os
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import pandas as pd
import numpy as np

In [ ]:
import sys
sys.path.append('..')
from tools.annotation.annotation import *

In [ ]:
import warnings
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
import numba
from numba.core.errors import NumbaDeprecationWarning, NumbaPendingDeprecationWarning
warnings.simplefilter("ignore", category=NumbaDeprecationWarning)
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
!pip install pyensembl

In [ ]:
!pyensembl install --release 77 --species mouse

In [ ]:
!pyensembl install --release 77 --species human

In [ ]:
!pyensembl list

In [ ]:
gtf = get_ens_dict('../../../Machine-learning-development-environment-for-single-cell-sequencing-data-analyses/api/tools/annotation/Mus_musculus.GRCm39.110.chr.gtf.gz')
gtf

In [ ]:
adata = sc.read_h5ad('../../../oscb/uploads/PBMC7K.h5ad')
adata

In [ ]:
adata.X.todense()

In [ ]:
ref1 = sc.read_h5ad('../../../oscb/uploads/PBMC7K_ref1.h5ad')
ref1

In [ ]:
ref2 = sc.read_h5ad('../../../oscb/uploads/PBMC7K_ref2.h5ad')
ref2

In [ ]:
adata.obs

In [ ]:
adata.obs.drop(["labels_ref2"], axis=1, inplace=True)

In [ ]:
adata.obs['split_idx'] = 'test'
adata

In [ ]:
ref1.obs['split_idx'] = 'train'
ref2.obs['split_idx'] = 'train'

In [ ]:
adata_ref1 = sc.concat([adata,ref1], join='outer')
adata_ref1

In [ ]:
adata_ref2 = sc.concat([adata,ref2], join='outer')
adata_ref2

In [ ]:
adata_ref1.write_h5ad('../../../oscb/uploads/PBMC7K_ref1a.h5ad', compression='gzip')
adata_ref2.write_h5ad('../../../oscb/uploads/PBMC7K_ref2a.h5ad', compression='gzip')

In [2]:
adata = sc.read_h5ad('../../../oscb/user_storage/Benchmarks/PBMC7Kannotation1_1757117752890/QC/PBMC7K_annotation1_scanpy.h5ad')

In [3]:
adata

AnnData object with n_obs × n_vars = 11989 × 2000
    obs: 'batch', 'labels', 'split_idx', 'n_genes', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_20_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'log1p_total_counts_hb', 'pct_counts_hb', 'outlier', 'doublet_score', 'predicted_doublet', 'clf_doublet', 'clf_score', 'leiden', 'louvain'
    var: 'n_cells', 'mt', 'ribo', 'hb', 'n_cells_by_counts', 'mean_counts', 'log1p_mean_counts', 'pct_dropout_by_counts', 'total_counts', 'log1p_total_counts', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'hvg', 'leiden', 'log1p', 'louvain', 'neighbors', 'pca', 'scrublet'
    obsm: 'X_pca', 'X_tsne', 'X_tsne_3D', 'X_umap', 'X_umap_3D'
    varm: 'PCs'
    layers: 'logCP10K', 'raw_counts'
    obsp: 'connectivities', 'distances'

In [14]:
adata.obs.labels.astype('category')

AAACCTGAGCTAGTGG-1        CD4 T cells
AAACCTGCACATTAGC-1        CD4 T cells
AAACCTGCACTGTTAG-1    CD14+ Monocytes
AAACCTGCATAGTAAG-1    CD14+ Monocytes
AAACCTGCATGAACCT-1        CD8 T cells
                           ...       
TTTGGTTTCGCTAGCG-1    CD14+ Monocytes
TTTGTCACACTTAACG-1        CD8 T cells
TTTGTCACAGGTCCAC-1           NK cells
TTTGTCAGTTAAGACA-1            B cells
TTTGTCATCCCAAGAT-1    CD14+ Monocytes
Name: labels, Length: 11989, dtype: category
Categories (9, object): ['B cells', 'CD4 T cells', 'CD8 T cells', 'CD14+ Monocytes', ..., 'FCGR3A+ Monocytes', 'Megakaryocytes', 'NK cells', 'Other']

In [15]:
adata.obs.labels.dtype.categories

Index(['B cells', 'CD4 T cells', 'CD8 T cells', 'CD14+ Monocytes',
       'Dendritic Cells', 'FCGR3A+ Monocytes', 'Megakaryocytes', 'NK cells',
       'Other'],
      dtype='object')

In [13]:
adata.obs.predicted_doublet.astype('category')

AAACCTGAGCTAGTGG-1    False
AAACCTGCACATTAGC-1    False
AAACCTGCACTGTTAG-1    False
AAACCTGCATAGTAAG-1    False
AAACCTGCATGAACCT-1    False
                      ...  
TTTGGTTTCGCTAGCG-1    False
TTTGTCACACTTAACG-1    False
TTTGTCACAGGTCCAC-1     True
TTTGTCAGTTAAGACA-1    False
TTTGTCATCCCAAGAT-1    False
Name: predicted_doublet, Length: 11989, dtype: category
Categories (2, bool): [False, True]

In [ ]:
adata.var

In [ ]:
adata.var.index.tolist()

In [ ]:
from pyensembl import EnsemblRelease
data = EnsemblRelease(77, species='human')

In [ ]:
data = EnsemblRelease(77, species='mouse')
data.index()

In [ ]:
symbol_ids = []
ensembl_ids = adata.var.index.tolist()
for ensembl_id in ensembl_ids:
    try:
        gene_name = data.gene_name_of_gene_id(ensembl_id)
        symbol_ids.append(gene_name)
    except ValueError:
        symbol_ids.append(ensembl_id) # Handle cases where ID is not found

print(symbol_ids)

In [ ]:
adata.var['symbol_ids'] = symbol_ids
adata.var

In [ ]:
is_ensembl('ISG15')

In [ ]:
adata.var['ensembl_ids'] = adata.var.index
adata.var = adata.var.set_index('symbol_ids')
adata.var

In [ ]:
adata.var_names[0]

In [ ]:
adata.X

In [ ]:
train_adata = adata[adata.obs.split_idx.str.contains('train'), :].copy()
test_adata = adata[adata.obs.split_idx.str.contains('test'), :].copy()

In [ ]:
import celltypist
import numpy as np
from celltypist import models

In [ ]:
ref_adata = train_adata
adata = test_adata
ref_adata = reset_x_to_raw(ref_adata)
sc.pp.filter_genes(ref_adata, min_cells = 10)
sc.pp.normalize_total(ref_adata, target_sum = 1e4) #Note this is only for cell annotation, recommended by authors but not best
sc.pp.log1p(ref_adata)

ref_adata = ref_adata[~ref_adata.obs[labels].isna()]
ref_model = celltypist.train(ref_adata, labels = labels, n_jobs = 4, use_SGD = False, feature_selection = True, top_genes = 300)
ref_predictions = celltypist.annotate(adata, model=ref_model, majority_voting=False)
ref_predictions_adata = ref_predictions.to_adata()
adata.obs["celltypist_ref_label"] = ref_predictions_adata.obs.loc[adata.obs.index, "predicted_labels"]
adata.obs["celltypist_ref_score"] = ref_predictions_adata.obs.loc[adata.obs.index, "conf_score"]

In [ ]:
adata_ref1.X.todense()

In [ ]:
import celltypist
from celltypist import models

In [ ]:
models.get_all_models()

In [ ]:
filename = os.path.basename('../../../oscb/user_storage/Benchmarks/facs-Bladder_1751302627486/QC/results/313b1738828fdf0d5157af2b12a71be6/facs_Bladder_MAGIC_imputation.h5ad')

In [ ]:
os.path.dirname('../../../oscb/user_storage/Benchmarks/facs-Bladder_1751302627486/QC/results/313b1738828fdf0d5157af2b12a71be6/facs_Bladder_MAGIC_imputation.h5ad')

In [ ]:
filename.split(".")[0]

In [ ]:
def run_celltypist(adata, model_name, refs = [], labels = None, species = 'mouse'):
    model = celltypist.Model.load(model_name)
    if species == 'mouse' and "Mouse" not in model_name:
        model.convert()
    adata = reset_x_to_raw(adata)

    sc.pp.filter_genes(adata, min_cells = 10)
    sc.pp.normalize_total(adata, target_sum=1e4) #not recommended for typical pp
    sc.pp.log1p(adata)
    
    if type(adata.X) != np.ndarray:
        adata.X = adata.X.toarray()
    
    predictions = celltypist.annotate(adata, model=model, majority_voting=True)
    predictions_adata = predictions.to_adata()
    adata.obs["celltypist_label"] = predictions_adata.obs.loc[adata.obs.index, "predicted_labels"]
    adata.obs["celltypist_score"] = predictions_adata.obs.loc[adata.obs.index, "conf_score"]

    if len(refs) > 0 and labels is not None:
        for input in refs:
            try:
                name = os.path.basename(input).split(".")[0]
                ref_ad = sc.read_h5ad(input)
                ref_ad = reset_x_to_raw(ref_ad)
                sc.pp.filter_genes(ref_ad, min_cells = 10)
                sc.pp.normalize_total(ref_ad, target_sum = 1e4) #Note this is only for cell annotation, recommended by authors but not best
                sc.pp.log1p(ref_ad)

                ref_ad = ref_ad[~ref_ad.obs[labels].isna()]
                ref_model = celltypist.train(ref_ad, labels = labels, n_jobs = 4, use_SGD = False, feature_selection = True, top_genes = 300)
                ref_predictions = celltypist.annotate(adata, model=ref_model, majority_voting=False)
                ref_predictions_adata = ref_predictions.to_adata()
                adata.obs["ref_"+name+"_label"] = ref_predictions_adata.obs.loc[adata.obs.index, "predicted_labels"]
                adata.obs["ref_"+name+"_score"] = ref_predictions_adata.obs.loc[adata.obs.index, "conf_score"]
            except Exception as e:
                print(e)
                continue

    return adata

In [ ]:
def reset_x_to_raw(adata, min_genes=200):
        if is_normalized(adata.X, min_genes) and not check_nonnegative_integers(adata.X):
            if "raw_counts" in adata.layers.keys():
                adata.layers["normalized_X"] = adata.X.copy()
                adata.X = adata.layers['raw_counts'].copy()
            elif adata.raw.X is not None:
                adata.layers["normalized_X"] = adata.X.copy()
                adata.X = adata.raw.X.copy()
            else:
                raise ValueError("Raw counts are not available.")
        
        return adata

In [ ]:
from typing import Optional, Union
import scipy.sparse as sp_sparse
from scipy.sparse import csr_matrix
import h5py
from anndata._core.sparse_dataset import SparseDataset
import jax
import jax.numpy as jnp

def is_normalized(expression_matrix, min_genes=200):
    if (not isinstance(expression_matrix, np.ndarray)):
        expression_matrix = expression_matrix.toarray()

    if np.min(expression_matrix) < 0 or np.max(expression_matrix) < min_genes:
        return True
    else:
        return False
        

def check_nonnegative_integers(
    data: Union[pd.DataFrame, np.ndarray, sp_sparse.spmatrix, h5py.Dataset],
    n_to_check: int = 20,
):
    """Approximately checks values of data to ensure it is count data."""
    # for backed anndata
    if isinstance(data, h5py.Dataset) or isinstance(data, SparseDataset):
        data = data[:100]

    if isinstance(data, np.ndarray):
        data = data
    elif issubclass(type(data), sp_sparse.spmatrix):
        data = data.data
    elif isinstance(data, pd.DataFrame):
        data = data.to_numpy()
    else:
        raise TypeError("data type not understood")

    ret = True
    if len(data) != 0:
        inds = np.random.choice(len(data), size=(n_to_check,))
        check = jax.device_put(data.flat[inds], device=jax.devices("cpu")[0])
        negative, non_integer = _is_not_count_val(check)
        ret = not (negative or non_integer)
    return ret

In [ ]:
adata = sc.read_h5ad('../../../oscb/user_storage/Benchmarks/facs-Bladder_1751302627486/QC/results/313b1738828fdf0d5157af2b12a71be6/facs_Bladder_MAGIC_imputation.h5ad')

In [ ]:
adata = run_celltypist(adata, model_name='Adult_Mouse_Gut.pkl', refs=['../../../oscb/user_storage/Benchmarks/facs-Bladder_1751302627486/QC/results/313b1738828fdf0d5157af2b12a71be6/facs_Bladder_MAGIC_imputation.h5ad'], labels='cell_ontology_class')

In [ ]:
adata.obs

In [ ]:
def scvi_transfer(adata, refs = [], labels = None):
    import scvi
    adata = reset_x_to_raw(adata)
    adata.obs['CellType'] = 'Unknown'
    adata.obs['Batch'] = 'Unknown'

    adatas = [sc.read_h5ad(input) for input in refs]
    dater = sc.concat(adatas, join='outer')
    sc.pp.filter_genes(dater, min_cells = 10)
    dater = dater[~dater.obs[labels].isna()]
    dater = reset_x_to_raw(dater)
    dater.obs['CellType'] = dater.obs[labels]
    dater.obs['Batch'] = 'reference'
    dater = sc.concat((adata, dater))
    dater.obs['Sample'] = dater.obs.index

    sc.pp.highly_variable_genes(dater, flavor = 'seurat_v3', n_top_genes=3000, batch_key="Batch", subset = True)
    scvi.model.SCVI.setup_anndata(dater, batch_key='Batch', categorical_covariate_keys = ['Sample'])
    vae = scvi.model.SCVI(dater)
    vae.train(max_epochs = 400, early_stopping = True)

    lvae = scvi.model.SCANVI.from_scvi_model(vae, adata = dater, unlabeled_category = 'Unknown',
                                        labels_key = 'CellType')

    lvae.train(max_epochs=20, n_samples_per_label=100)

    dater.obs['scVI_predicted'] = lvae.predict(dater)
    dater.obs['scVI_transfer_score'] = lvae.predict(soft = True).max(axis = 1)
    dater = dater[dater.obs.Batch == 'Unknown']

    adata.obs = adata.obs.merge(right = dater.obs[['scVI_predicted', 'scVI_transfer_score']], left_index=True, right_index=True)
    adata.obs = adata.obs.drop('CellType', axis=1) 

    return adata

@jax.jit
def _is_not_count_val(data: jnp.ndarray):
    negative = jnp.any(data < 0)
    non_integer = jnp.any(data % 1 != 0)

    return negative, non_integer

In [ ]:
adata = scvi_transfer(adata, refs=['../../../oscb/user_storage/Benchmarks/facs-Bladder_1751302627486/QC/results/313b1738828fdf0d5157af2b12a71be6/facs_Bladder_MAGIC_imputation.h5ad'], labels='cell_ontology_class')

In [ ]:
adata.obs